# RAG — Sample Code (reference template)
### Module 4 sample notebook

A minimal, framework-free RAG pipeline you can copy into your own project. Runs offline with `sentence-transformers` + `chromadb`; generation uses OpenAI if `OPENAI_API_KEY` is set, else it prints the prompt. See `embeddings_and_vector_db.ipynb` and `rag_example.ipynb` for the full explanations.

### 1. Build the knowledge base (load → chunk → embed → store)

In [ ]:
# pip install sentence-transformers chromadb
import os
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def chunk(text, size=600, overlap=100):
    pieces, start = [], 0
    while start < len(text):
        piece = text[start:start + size].strip()
        if piece:
            pieces.append(piece)
        start += size - overlap
    return pieces

# load + chunk the documents
chunks, sources = [], []
for fname in ["ssrf.txt", "cybersecurity_kb.md"]:
    if os.path.exists(fname):
        for piece in chunk(open(fname, encoding="utf-8").read()):
            chunks.append(piece); sources.append(fname)

# embed + store in a Chroma collection
client = chromadb.EphemeralClient()
kb = client.get_or_create_collection("kb", metadata={"hnsw:space": "cosine"})
kb.add(ids=[f"c{i}" for i in range(len(chunks))],
       documents=chunks,
       embeddings=embedder.encode(chunks).tolist(),
       metadatas=[{"source": s} for s in sources])
print("Indexed", kb.count(), "chunks.")

### 2. Ask (retrieve → augment → generate)

In [ ]:
def ask(question, k=4):
    # retrieve
    hits = kb.query(query_embeddings=embedder.encode([question]).tolist(), n_results=k)
    context = "\n\n".join(hits["documents"][0])
    # augment
    system = ("Answer using ONLY the context. If it isn't there, say you don't know.\n\n"
              "Context:\n" + context)
    # generate
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": question}])
        return r.choices[0].message.content
    return "[no key] Prompt assembled with retrieved context (would go to the LLM):\n" + system[:400] + " ..."

print(ask("What is SSRF and how is it exploited?"))